In [ ]:
import keras
import numpy as np
import whisper
import sentencepiece as spm


mediumModel = whisper.load_model('medium')

model = keras.models.load_model("seitsemanVeljesta_best_model.keras")

sp = spm.SentencePieceProcessor()
sp.load('seitsemanVeljesta_sp.model')

def generate_text(model, sp, start_text, length=100, seq_length=32):
    input_ids = sp.encode_as_ids(start_text)
    generated_ids = input_ids[:]

    for _ in range(length):
        x = generated_ids[-seq_length:]
        if len(x) < seq_length:
            x = [0] * (seq_length - len(x)) + x

        x = np.array(x, dtype=np.int32)[None, :]  # ensure int32 for TensorFlow
        preds = model.predict(x, verbose=0)
        next_id = int(np.argmax(preds[0, -1]))  # make sure it's an int
        generated_ids.append(next_id)

    return sp.decode_ids([int(i) for i in generated_ids])

fileName = input("Please enter audio file name:")
fileType = input("Please enter file type:")

result = mediumModel.transcribe(f"{fileName}.{fileType}", language="fi")
transcribed_text = result["text"]

print("Transkriptio:")
print(transcribed_text)

generated = generate_text(model, sp, transcribed_text, length=80)
print("\nGeneroitu teksti:")
print(generated)